# 10 --- Results Interpretation

Run a full DocuFlow pipeline and inspect everything the result contains: extracted data, field-level evidence, OCR confidence, consensus, validation errors, review reasons, trace events, and serialization output.

This notebook uses OCR so you can see confidence data. By default DocuFlow preserves the exact source text for textual fields; set `normalize_output=True` only when you want canonicalized values such as ISO date strings. If you switch the parser to `docling`, the `raw_text` field will contain markdown instead of plain text.

## What this notebook covers

- End-to-end pipeline execution
- Top-level result scores and metadata
- Field-level evidence, OCR, consensus, and verification
- Validation and review metadata
- Trace inspection
- Provenance and JSON serialization

The goal is to make it obvious where every piece of information lives in the final result object.

In [ ]:
import json
import os
from pprint import pprint

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "<set-your-gemini-api-key>")

PDF_PATH = "data/sample_invoice.pdf"

from pydantic import BaseModel, Field
from typing import Optional, List


class LineItem(BaseModel):
    description: str = Field(description="Line item description")
    quantity: Optional[float] = Field(default=None, description="Quantity ordered")
    unit_price: Optional[float] = Field(default=None, description="Price per unit")
    tax_rate: Optional[float] = Field(default=None, description="Tax rate percentage for this item")
    amount: float = Field(description="Line item total amount")


class Invoice(BaseModel):
    supplier_name: str = Field(description="Name of the supplier or vendor")
    invoice_number: str = Field(description="Invoice reference number")
    invoice_date: str = Field(description="Date the invoice was issued exactly as written")
    due_date: Optional[str] = Field(default=None, description="Payment due date exactly as written")
    po_number: Optional[str] = Field(default=None, description="Purchase order number")
    currency: str = Field(default="USD", description="Currency code (ISO 4217)")
    bill_to_company: Optional[str] = Field(default=None, description="Company billed")
    ship_to_company: Optional[str] = Field(default=None, description="Company shipped to")
    subtotal: Optional[float] = Field(default=None, description="Amount before tax")
    tax_rate: Optional[float] = Field(default=None, description="Tax rate as percentage")
    tax_amount: Optional[float] = Field(default=None, description="Tax amount")
    total: float = Field(description="Total amount including tax")
    payment_terms: Optional[str] = Field(default=None, description="Payment terms")
    line_items: Optional[List[LineItem]] = Field(default=None, description="Individual line items")


print(f"Schema fields: {len(Invoice.model_fields)}")
for name, field in Invoice.model_fields.items():
    required = "required" if field.is_required() else "optional"
    print(f"  {name:<20} {required}")

In [ ]:
from docuflow import DocumentPipeline
from docuflow.validation import RequiredFields, EvidenceRequired
from docuflow.review import OverallConfidenceBelow, FieldMissing, NoEvidence

pipeline = DocumentPipeline(
    parser="tesseract",
    model="gemini/gemini-2.5-flash",
    extraction_type="auto",
    extraction_mode="multi",
    n_instances=3,
    normalize_output=False,  # default: preserve exact source text
    storage={"type": "local", "path": "./.docuflow_store_results_interpretation"},
    validators=[
        RequiredFields(["supplier_name", "invoice_number", "invoice_date", "total"]),
        EvidenceRequired(["total", "invoice_number"]),
    ],
    review_rules=[
        OverallConfidenceBelow(0.95),
        FieldMissing(["total", "invoice_number"]),
        NoEvidence(["total"]),
    ],
    verification={
        "trigger_consensus_below": 0.75,
        "trigger_ocr_below": 0.65,
        "include_unmatched": True,
        "max_fields": 5,
        "dpi": 300,
        "apply_corrections": True,
    },
    escalation={
        "min_ocr_score": 0.6,
        "max_low_confidence_ratio": 0.4,
        "escalate_to": "vision",
    },
    context="You are extracting invoice data for accounting review.",
)

print("Pipeline configured.")
print("Steps are implicit in DocumentPipeline: ingest -> parse -> extract_auto -> verify -> validate -> review -> store")

## Run the pipeline

`DocumentPipeline.run_sync()` returns the final `ExtractionResult`. Internally the pipeline also records a trace, which is attached to the result as `result.trace`.

In [ ]:
result = pipeline.run_sync(PDF_PATH, schema=Invoice)

print("Extraction complete.")
print(f"Result type: {type(result).__name__}")
print(f"Document ID: {result.document_id}")

## Top-level result summary

These are the fields most users want first: the extracted payload, overall trust, the OCR score, the consensus score, review status, and the model/parser names.

In [ ]:
summary = {
    "document_id": result.document_id,
    "schema_name": result.schema_name,
    "model_name": result.model_name,
    "parser_name": result.parser_name,
    "trace_id": result.trace_id,
    "confidence": result.confidence,
    "confidence_score": result.confidence_score,
    "consensus_score": result.consensus_score,
    "escalated": result.escalated,
    "escalation_reason": result.escalation_reason,
    "needs_review": result.needs_review,
    "review_status": result.review_status,
    "review_reasons": result.review_reasons,
    "validation_errors": result.validation_errors,
    "usage": result.usage.model_dump() if result.usage else None,
}

pprint(summary)

## Extracted data

`result.data` is the validated plain-data payload. `result.fields` carries the richer per-field records.

In [ ]:
print("=== result.data ===")
print(json.dumps(result.data, indent=2, default=str))

print("\n=== result.fields keys ===")
print(list(result.fields.keys()))

In [ ]:
print("=== Field details ===")
for name, field in result.fields.items():
    print(f"\n[{name}]")
    print(f"  value: {field.value!r}")
    print(f"  original_value: {field.original_value!r}")
    print(f"  confidence: {field.confidence:.4f}")
    print(f"  validation_status: {field.validation_status}")
    if field.errors:
        print(f"  errors: {field.errors}")
    if field.trust is not None:
        print(
            "  trust: "
            f"score={field.trust.score:.4f}, "
            f"agreement={field.trust.agreement}, "
            f"found_in_source={field.trust.found_in_source}, "
            f"auto_accept={field.trust.auto_accept}"
        )
    if field.ocr is not None:
        print(
            "  ocr: "
            f"score={field.ocr.score}, "
            f"match_method={field.ocr.match_method}, "
            f"match_ratio={field.ocr.match_ratio}, "
            f"page={field.ocr.page_number}"
        )
        if field.ocr.matched_text:
            print(f"  ocr.matched_text: {field.ocr.matched_text!r}")
    if field.consensus is not None:
        print(
            "  consensus: "
            f"agreement={field.consensus.agreement}, "
            f"ratio={field.consensus.agreement_ratio}, "
            f"majority_ratio={field.consensus.majority_ratio}, "
            f"instances={field.consensus.n_succeeded}/{field.consensus.n_instances}"
        )
    if field.verification is not None:
        print(
            "  verification: "
            f"verified={field.verification.verified}, "
            f"agrees={field.verification.agrees}, "
            f"changed={field.verification.changed}, "
            f"reason={field.verification.reason!r}"
        )
    if field.evidence:
        ev = field.evidence[0]
        print(
            "  evidence[0]: "
            f"page={ev.page_number}, "
            f"confidence={ev.confidence}, "
            f"text={ev.text[:120]!r}"
        )

## Raw text, validation, review, and errors

`result.raw_text` holds the parsed text for text-based pipelines. When you use the Docling parser, this attribute contains markdown.

Validation errors, review reasons, and corrections are all kept on the final result so downstream code can decide how to handle them without re-running the pipeline.

In [ ]:
print("=== raw_text preview ===")
print(result.raw_text[:2500])

print("\n=== validation_errors ===")
print(json.dumps(result.validation_errors, indent=2, default=str))

print("\n=== review_reasons ===")
print(result.review_reasons)

print("\n=== review_verdicts ===")
for verdict in result.review_verdicts:
    print(verdict.model_dump())

print("\n=== corrections ===")
for correction in result.corrections:
    print(correction.model_dump())

## Trace inspection

The trace shows the pipeline timeline, including each step and its timing. This is useful when you want to debug slow runs, failed runs, or verify which stages actually executed.

In [ ]:
trace = result.trace

print(f"Trace ID: {trace.trace_id if trace else result.trace_id}")
print(f"Started at: {trace.started_at if trace else 'n/a'}")
print(f"Completed at: {trace.completed_at if trace else 'n/a'}")

print("\n=== Trace events ===")
for event in trace.events if trace else []:
    meta = event.metadata or {}
    print(
        f"{event.timestamp:%Y-%m-%d %H:%M:%S} | "
        f"{event.step_name:<16} | "
        f"{event.event_type:<18} | "
        f"{(event.duration_ms or 0):>8.2f} ms | "
        f"{meta}"
    )

## Provenance and serialization

Use `provenance()` when you need a compact per-field audit view. Use `model_dump_json()` when you want to persist or transmit the full structured result.

In [ ]:
prov = result.provenance("total")
print("=== provenance('total') ===")
pprint(prov["total"].model_dump())

print("\n=== JSON serialization preview ===")
json_blob = result.model_dump_json(indent=2)
print(json_blob[:4000])
print("...")

## Failure pattern

If `pipeline.run_sync(...)` raises `WorkflowError`, the exception carries the partial result object. Inspect `exc.result.errors`, `exc.result.state`, and `exc.result.trace` to understand where the pipeline stopped.